# Exercise 4 — aggregate_sentiment and sentiment_to_signal

`aggregate_sentiment` reduces a list of headline scores to a single daily sentiment number by taking the mean. `sentiment_to_signal` applies a threshold: positive enough sentiment → long (1), otherwise flat (0). The threshold default of 0.1 means the model needs to see a mild net positive signal to go long — not just any non-zero score.

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]
def parse_score(text):
    """Extract and clamp a float from LLM output. Returns 0.0 if not found."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return 0.0
    return max(-1.0, min(1.0, float(matches[0])))

def build_sentiment_prompt(headline):
    return [
        {
            "role": "system",
            "content": (
                "You are a financial news sentiment analyzer. "
                "Score the sentiment from -1.0 (very bearish) to 1.0 (very bullish). "
                "Reply with ONLY a single decimal number. No explanation."
            ),
        },
        {"role": "user", "content": f"Headline: {headline}"},
    ]
def score_headline(headline, llm_fn=None):
    messages = build_sentiment_prompt(headline)
    if llm_fn is not None:
        response = llm_fn(messages)
    else:
        import ollama
        response = ollama.chat(model="llama3.2", messages=messages)["message"]["content"]
    return parse_score(response)
def score_headlines(headlines, llm_fn=None):
    return [score_headline(h, llm_fn) for h in headlines]

def aggregate_sentiment(scores):
    """Mean of a list of sentiment scores, clamped to [-1.0, 1.0].

    Returns 0.0 for an empty list (neutral default).

    Implementation:
        if not scores: return 0.0
        return max(-1.0, min(1.0, sum(scores) / len(scores)))
    """
    # TODO: implement
    return 0.0


def sentiment_to_signal(sentiment, threshold=0.1):
    """Convert aggregate sentiment to a binary trading signal.

    Returns 1 (long) if sentiment > threshold, else 0 (flat).

    Implementation:
        return 1 if sentiment > threshold else 0
    """
    # TODO: one line
    return 0


### Checks

In [ ]:
checks = 0

# 1 — aggregate_sentiment returns the mean of the scores
try:
    scores = [0.4, 0.6, 0.2]
    agg = aggregate_sentiment(scores)
    assert abs(agg - (0.4+0.6+0.2)/3) < 1e-9, f"expected {(0.4+0.6+0.2)/3:.4f}, got {agg}"
    checks += 1; print("✅ 1 aggregate_sentiment returns correct mean")
except Exception as e:
    print("❌ 1:", e)

# 2 — aggregate_sentiment: empty list → 0.0
try:
    assert aggregate_sentiment([]) == 0.0, f"expected 0.0 for empty, got {aggregate_sentiment([])}"
    checks += 1; print("✅ 2 aggregate_sentiment returns 0.0 for empty list")
except Exception as e:
    print("❌ 2:", e)

# 3 — aggregate_sentiment: clamps the mean to [-1.0, 1.0]
try:
    # This shouldn't happen with valid scores, but handle edge cases
    agg = aggregate_sentiment([0.9, 0.9, 0.9])
    assert -1.0 <= agg <= 1.0, f"out of range: {agg}"
    checks += 1; print("✅ 3 aggregate_sentiment result is within [-1.0, 1.0]")
except Exception as e:
    print("❌ 3:", e)

# 4 — sentiment_to_signal: positive above threshold → 1
try:
    assert sentiment_to_signal(0.5)  == 1, "0.5 > 0.1 → should be 1"
    assert sentiment_to_signal(0.1)  == 0, "0.1 is NOT > 0.1 → should be 0"
    assert sentiment_to_signal(0.11) == 1, "0.11 > 0.1 → should be 1"
    checks += 1; print("✅ 4 sentiment_to_signal: >threshold → 1; ≤threshold → 0")
except Exception as e:
    print("❌ 4:", e)

# 5 — full pipeline: bullish headlines → signal = 1
try:
    scores = score_headlines(BULLISH_HEADLINES, llm_fn=_mock_llm)
    agg    = aggregate_sentiment(scores)
    sig    = sentiment_to_signal(agg)
    assert sig == 1, f"bullish headlines should give signal=1, got sig={sig} (agg={agg:.3f})"
    print(f"  Bullish: mean score={agg:.3f} → signal={sig}")

    scores = score_headlines(BEARISH_HEADLINES, llm_fn=_mock_llm)
    agg    = aggregate_sentiment(scores)
    sig    = sentiment_to_signal(agg)
    assert sig == 0, f"bearish headlines should give signal=0, got sig={sig} (agg={agg:.3f})"
    print(f"  Bearish: mean score={agg:.3f} → signal={sig}")
    checks += 1; print("✅ 5 full pipeline: bullish→1, bearish→0")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
